In [ ]:
"""
Fix final: grafico_3_setor_v2.py
Corrige dicionário VD4010 — códigos são dezenas (10, 20, 30...)
"""

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE       = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
ARQUIVO_IN = BASE / "data/processed/pnadc_2023_anual.csv"
DIR_FIGS   = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

In [ ]:
df = pd.read_csv(ARQUIVO_IN, dtype=str)
df["VD4031"] = pd.to_numeric(df["VD4031"], errors="coerce")
ocupados = df[df["VD4002"].str.strip() == "1"].copy()
ocupados["VD4010"] = ocupados["VD4010"].str.strip()

print(f"Ocupados: {len(ocupados):,}")

In [ ]:
setores = {
    "00": "Ativ. mal definidas",
    "10": "Agropecuária",
    "20": "Indústria geral",
    "30": "Construção",
    "40": "Comércio e rep.",
    "50": "Transp. e armaz.",
    "60": "Alojamento e alim.",
    "70": "Informação e serv. prof.",
    "80": "Adm. pública e educação",
    "81": "Saúde",
    "90": "Serv. domésticos e outros",
}

ocupados["setor_nome"] = ocupados["VD4010"].map(setores)

media_setor = (
    ocupados.groupby("setor_nome")["VD4031"]
    .mean()
    .dropna()
    .sort_values(ascending=True)
)

print("\nMédia de horas por setor:")
print(media_setor.round(1).to_string())

In [ ]:
cores_setor = [
    "#E84545" if v > 44 else "#F5A623" if v > 40 else "#2C6FBF"
    for v in media_setor.values
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(media_setor.index, media_setor.values, color=cores_setor, height=0.6)

ax.axvline(40, color="#2C6FBF", linewidth=1.5, linestyle="--", alpha=0.8, label="40h (proposta)")
ax.axvline(44, color="#F5A623", linewidth=1.5, linestyle="--", alpha=0.8, label="44h (CLT atual)")

for bar, val in zip(bars, media_setor.values):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}h", va="center", fontsize=10
    )

ax.set_xlabel("Média de horas habituais semanais", fontsize=11)
ax.set_title(
    "Jornada média semanal por setor de atividade — Brasil 2023\n"
    "Trabalhadores ocupados · Fonte: PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)
ax.set_xlim(0, 58)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(DIR_FIGS / "03_jornada_media_por_setor.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 3 salvo.")